# PrismaCoder - Inference with Automatic Validation & Repair

Natural language (+ optional schema) -> generated artifact -> gate validation -> auto-repair -> final code. Uses the repo's `PrismaCoderPipeline` + `RepairLoop` (report Part 14).


In [ ]:
# Cell 1 - setup: clone the repo for the package code and install deps
import os, subprocess, sys

REPO_DIR = '/content/PrismaCoder'  # or any writable path
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/prismacoder/PrismaCoder.git', REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

MODEL_ID = 'prismacoder/PrismaCoder-3B'   # or runs/prismacoder-3b-qlora-sft/merged
USE_FULL_GATES = False                    # True when Node >= 18.18 is available
REPAIR_ATTEMPTS = 3


In [ ]:
# Cell 2 - load the model (merged checkpoint from HF or local dir)
from prismacoder.inference.pipeline import PrismaCoderPipeline

pipe = PrismaCoderPipeline.from_local(MODEL_ID, use_full_gates=USE_FULL_GATES,
                                      repair_attempts=REPAIR_ATTEMPTS)
print('pipeline ready')


In [ ]:
# Cell 3 - single-shot inference with the validate+repair loop
SCHEMA = '''\
datasource db {
  provider = "postgresql"
  url      = env("DATABASE_URL")
}

model Order {
  id        Int      @id @default(autoincrement())
  status    String   @default("PENDING")
  totalCents Int     @default(0)
  userId    Int
  createdAt DateTime @default(now())
}

model User {
  id     Int     @id @default(autoincrement())
  email  String  @unique
  orders Order[]
}'''

INSTRUCTION = 'List the 20 most recent orders with their users, newest first, excluding cancelled.'

result = pipe.run(INSTRUCTION, SCHEMA, repair=True)
print(result.artifact)
print('---')
print('status:', result.status, '| attempts:', len(result.attempts),
      '| final gate:', result.error_summary['final_gate'] or '-')


In [ ]:
# Cell 4 - inspect the repair trace (gate verdicts + E1-E10 classes)
for a in result.attempts:
    print(f"attempt {a.attempt}: {a.status} via {a.gate} (class {a.error_class or '-'})")
    for d in a.diagnostics[:3]:
        print('   -', d)


In [ ]:
# Cell 5 - interactive helper: try your own request
from IPython.display import display, Markdown  # type: ignore

def ask(instruction: str, schema: str = '', repair: bool = True):
    res = pipe.run(instruction, schema, repair=repair)
    print(res.artifact)
    print('status:', res.status)
    return res

# res = ask('Create a user with email a@b.com and an order of 1999 cents atomically.', SCHEMA)


## Notes

- With `USE_FULL_GATES=True` and Node >= 18.18, repair targets the real V1-V5 chain (prisma validate/generate, tsc, db push, vitest) instead of the static validator.
- The API server exposes the same behavior: `uvicorn inference.server.app:app` (`POST /v1/generate`, `POST /v1/generate/stream`).
